# JAX-MD: Running a Simulation

This notebook demonstrates how to run a coarse-grained DNA simulation using JAX-MD as the simulation backend.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

In this notebook we run a DNA simulation using [JAX-MD](https://github.com/jax-md/jax-md) as the backend. JAX-MD provides GPU-accelerated, fully-differentiable molecular dynamics — making it possible to backpropagate through the entire simulation trajectory.

## Imports

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import jax_md

import mythos.energy.dna1 as dna1_energy
import mythos.input.topology as jdna_top
import mythos.input.trajectory as jdna_traj
import mythos.simulators.jax_md as jdna_jaxmd

jax.config.update("jax_enable_x64", True)

## Configuration

In [ ]:
N_STEPS = 5_000
EXPERIMENT_DIR = Path("../../data/sys-defs/simple-helix")

## Load topology and initial positions

In [ ]:
topology = jdna_top.from_oxdna_file(EXPERIMENT_DIR / "sys.top")
initial_positions = (
    jdna_traj.from_file(
        EXPERIMENT_DIR / "bound_relaxed.conf",
        topology.strand_counts,
    )
    .states[0]
    .to_rigid_body()
)

## Build the energy function and simulator

`create_default_energy_fn` assembles the full oxDNA1 composed energy function. We then pass it to `JaxMDSimulator` along with the simulation parameters.

In [ ]:
energy_fn = dna1_energy.create_default_energy_fn(
    topology=topology,
    displacement_fn=jax_md.space.free()[0],
)

experiment_config, _ = dna1_energy.default_configs()

dt = experiment_config["dt"]
kT = experiment_config["kT"]
diff_coef = experiment_config["diff_coef"]
rot_diff_coef = experiment_config["rot_diff_coef"]

gamma = jax_md.rigid_body.RigidBody(
    center=jnp.array([kT / diff_coef], dtype=jnp.float64),
    orientation=jnp.array([kT / rot_diff_coef], dtype=jnp.float64),
)
mass = jax_md.rigid_body.RigidBody(
    center=jnp.array([experiment_config["nucleotide_mass"]], dtype=jnp.float64),
    orientation=jnp.array([experiment_config["moment_of_inertia"]], dtype=jnp.float64),
)

params = energy_fn.opt_params()

simulator = jdna_jaxmd.JaxMDSimulator(
    energy_fn=energy_fn,
    simulator_params=jdna_jaxmd.StaticSimulatorParams(
        seq=jnp.array(topology.seq),
        mass=mass,
        bonded_neighbors=topology.bonded_neighbors,
        checkpoint_every=100,
        dt=dt,
        kT=kT,
        gamma=gamma,
    ),
    space=jax_md.space.free(),
    simulator_init=jax_md.simulate.nvt_langevin,
    neighbors=jdna_jaxmd.NoNeighborList(unbonded_nbrs=topology.unbonded_neighbors),
)

## Run the simulation

In [ ]:
key = jax.random.PRNGKey(0)
sim_fn = jax.jit(lambda opts: simulator.run(opts, initial_positions, N_STEPS, key))

print("Running simulation...")
trajectory = sim_fn(params)
print(f"Simulation Complete! \u2705 Trajectory length: {trajectory.center.shape[0]}")